# OmniVoice Quick Start — Robust Long-form Fork

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/binhminhanh1235/OmniVoice/blob/fix/robust-longform-tts/docs/OmniVoice.ipynb)

This notebook uses the patched fork [binhminhanh1235/OmniVoice](https://github.com/binhminhanh1235/OmniVoice/tree/fix/robust-longform-tts).

The branch includes the silence/edge fix from upstream PR #259 plus an opt-in robust long-form pipeline for reducing skipped words, repeated phrases, sentence mixing, and clipped phonemes.

**Contents:**
1. Installation from the patched GitHub branch
2. Option A — Gradio Demo
3. Option B — Python API
   - 3.1 Load Model
   - 3.2 Voice Cloning
   - 3.3 Voice Design
   - 3.4 Auto Voice
4. Robust Long-form Generation


## 1. Installation

Install OmniVoice directly from your patched GitHub branch instead of the PyPI release.

The model weights are still loaded from `k2-fsa/OmniVoice` on Hugging Face; only the Python code comes from your fork.


In [ ]:
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@fix/robust-longform-tts"

import omnivoice
print("OmniVoice package:", omnivoice.__version__)
print("Code branch: binhminhanh1235/OmniVoice@fix/robust-longform-tts")


## 2. Option A — Gradio Demo

Launch the CLI/demo from the patched package installed above.

> The robust ASR verification pipeline in section 4 is available through the Python API. For long narration where exact wording matters, prefer section 4.


In [ ]:
!omnivoice-demo --share

## 3. Option B — Python API

### 3.1 Load Model

The code is imported from your GitHub branch, while model weights are downloaded from the official Hugging Face checkpoint.


In [ ]:
from omnivoice import (
    OmniVoice,
    OmniVoiceGenerationConfig,
    RobustLongFormConfig,
    RobustLongFormGenerator,
)
import soundfile as sf
import torch
from IPython.display import Audio, display

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,

    # Used for optional reference transcription and robust verification.
    # Keep ASR on CPU so it does not compete with OmniVoice for T4 VRAM.
    load_asr=True,
    asr_model_name="openai/whisper-small.en",
    asr_device="cpu",
)

print("Sampling rate:", model.sampling_rate)


### 3.2 Voice Cloning

Upload a clean 3–10 second reference clip.

For best stability, provide an exact transcript in `REF_TEXT`. If `REF_TEXT` is empty, the CPU Whisper model will auto-transcribe the reference.

A single reusable `VoiceClonePrompt` is created once and reused by both short-form and robust long-form generation.


In [ ]:
from google.colab import files

print("Upload a reference audio file (wav/mp3/flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"Uploaded: {ref_audio_path}")

# Strongly recommended: paste the exact transcript of the reference audio.
# Leave empty only if you want Whisper to auto-transcribe it.
REF_TEXT = ""

voice_prompt = model.create_voice_clone_prompt(
    ref_audio=ref_audio_path,
    ref_text=REF_TEXT.strip() or None,
    preprocess_prompt=True,
)

print("Reference text used by OmniVoice:")
print(voice_prompt.ref_text)


In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice cloning.",
    language="en",
    voice_clone_prompt=voice_prompt,
)

sf.write("clone_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))


### 3.3 Voice Design

Describe the desired voice with speaker attributes — no reference audio needed.

Supported attributes include gender, age, pitch, style, English accent, and Chinese dialect. See [docs/voice-design.md](https://github.com/binhminhanh1235/OmniVoice/blob/fix/robust-longform-tts/docs/voice-design.md).


In [ ]:
audio = model.generate(
    text="Hello, this is a test of zero-shot voice design.",
    instruct="female, low pitch, british accent",
)

sf.write("design_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))


### 3.4 Auto Voice

Let the model choose a voice automatically — no reference audio or instruct needed.

In [ ]:
audio = model.generate(
    text="This is a sentence generated with automatic voice selection.",
)

sf.write("auto_out.wav", audio[0], model.sampling_rate)
display(Audio(audio[0], rate=model.sampling_rate))


## 4. Robust Long-form Generation

Use this path for narration where wording must remain faithful.

The wrapper:

- cleans HTML entities such as `&#x20;`;
- chunks at paragraph/sentence boundaries before considering commas;
- reuses the same `VoiceClonePrompt`;
- disables nested automatic chunking;
- avoids speech crossfade/fades at chunk edges;
- transcribes each generated chunk with Whisper;
- rejects missing words, repeated n-grams, strong word-count drift, and missing semantic negations such as `not`, `no`, `never`, `without`;
- retries only the failed chunk;
- recursively splits a persistent failure into smaller semantic pieces;
- stitches accepted chunks with explicit silence.


In [ ]:
robust_config = RobustLongFormConfig(
    max_chunk_words=24,
    max_chunk_chars=220,

    max_retries=3,
    max_split_depth=2,

    verify_with_asr=True,
    asr_model_name="openai/whisper-small.en",
    asr_device="cpu",

    max_wer=0.18,
    min_similarity=0.82,
    min_word_ratio=0.74,
    max_word_ratio=1.30,

    pause_ms=320,
    paragraph_pause_ms=460,

    # False: after exhausting retries/splits, keep the best candidate with warning.
    # True: raise instead of accepting any unverified chunk.
    strict=False,
)

robust_generator = RobustLongFormGenerator(
    model,
    robust_config,
)

generation_config = OmniVoiceGenerationConfig(
    num_step=32,
    guidance_scale=2.0,
    position_temperature=1.0,
    class_temperature=0.0,

    # RobustLongFormGenerator already splits semantic chunks.
    audio_chunk_threshold=1e9,

    # Do not attenuate low-level attacks/releases at chunk edges.
    pad_duration=0.0,
    fade_duration=0.0,

    # PR #259 silence/edge controls.
    postprocess_output=True,
    output_min_silence_ms=650,
    output_keep_silence_ms=180,
    output_lead_silence_ms=80,
    output_trail_silence_ms=130,
    output_target_lead_silence_ms=0,
    output_target_trail_silence_ms=0,
)

print("Robust long-form pipeline ready.")


In [ ]:
TEXT = """Let's be clear from the beginning.

This is not about refusing kindness to someone who is sick, grieving, poor, overwhelmed, or genuinely trying to rebuild their life.

This is not about becoming suspicious of everyone who needs help.

And this is not about calling people "toxic" because they disappointed you.

This is about repeated patterns.

Patterns that reject truth.
Patterns that avoid responsibility.
Patterns that turn compassion into permission.

Love is not unlimited access.

Forgiveness is not instant trust.

And helping a person is not the same as helping the pattern that keeps hurting them, others, and sometimes you.

So as we walk through these five patterns, do not use them to judge someone quickly. Use them first to examine the kind of help you are giving.
""".strip()

result = robust_generator.generate(
    TEXT,
    language="en",
    voice_clone_prompt=voice_prompt,
    generation_config=generation_config,
)

OUTPUT_PATH = "/content/omnivoice_robust_output.wav"
sf.write(
    OUTPUT_PATH,
    result.audio,
    result.sampling_rate,
)

print("All chunks verified:", result.all_verified)
print("Final chunks:", len(result.chunks))
print("Output:", OUTPUT_PATH)

display(Audio(result.audio, rate=result.sampling_rate))


In [ ]:
# Inspect verification results.
import pandas as pd

report_df = pd.DataFrame([
    {
        "chunk": i,
        "accepted": r.accepted,
        "attempts": r.attempts,
        "depth": r.depth,
        "wer": round(r.wer, 4),
        "similarity": round(r.similarity, 4),
        "word_ratio": round(r.word_ratio, 4),
        "critical_missing": ", ".join(r.critical_missing),
        "extra_repetitions": ", ".join(r.extra_repetitions),
        "expected": r.text,
        "whisper_transcript": r.transcript,
    }
    for i, r in enumerate(result.reports, 1)
])

display(report_df)

REPORT_PATH = "/content/omnivoice_robust_report.csv"
report_df.to_csv(REPORT_PATH, index=False)
print("Report:", REPORT_PATH)


In [ ]:
# Download the generated WAV and verification report.
from google.colab import files

files.download(OUTPUT_PATH)
files.download(REPORT_PATH)
